# Ride Demand Forecasting Data Prep Engine
**Step-by-step beginner project using your uploaded data.** Run cells in order. No `sqlite3`, `os`, or `pathlib` in notebook. The supplied SQL INSERT statements were converted to `city_zones.csv` without changing their values.

**Data limitations:** `ride_date` has dates but no time, so real peak-hour flags and hourly demand cannot be calculated. The dataset has an existing `surge_flag`, which is preserved. No ride-level cancellation status exists; cancellation rate comes from rider account totals. The numeric traffic index is mapped to Low/Medium/High using illustrative cutoffs, not source-provided labels.

## 1. Load libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from scipy.stats import zscore

## 2. Load CSV, JSON and converted SQL data

In [ ]:
riders = pd.read_csv("riders.csv")
trips = pd.read_json("trips.json")
zones = pd.read_csv("city_zones.csv")

### Riders: first rows, info, missing, duplicates

In [ ]:
display(riders.head())
riders.info()
print("Missing:\n", riders.isnull().sum())
print("Duplicates:", riders.duplicated().sum())

### Trips: first rows, info, missing, duplicates

In [ ]:
display(trips.head())
trips.info()
print("Missing:\n", trips.isnull().sum())
print("Duplicates:", trips.duplicated().sum())

### Zones: first rows, info, missing, duplicates

In [ ]:
display(zones.head())
zones.info()
print("Missing:\n", zones.isnull().sum())
print("Duplicates:", zones.duplicated().sum())

## 3. Save before-cleaning measurements

In [ ]:
rows_before = len(trips)
missing_before = riders.isna().sum().sum() + trips.isna().sum().sum() + zones.isna().sum().sum()
print("Invalid age:", (riders["age"] < 10).sum())
print("Negative distance:", (trips["distance_km"] < 0).sum())
print("Negative fare:", (trips["fare_amount"] < 0).sum())

## 4. Remove duplicate records

In [ ]:
riders = riders.drop_duplicates().copy()
trips = trips.drop_duplicates().copy()
zones = zones.drop_duplicates().copy()

## 5. Invalid values: remove unrealistic records

In [ ]:
riders.loc[riders["age"] < 10, "age"] = np.nan
trips = trips[(trips["distance_km"].isna()) | (trips["distance_km"] >= 0)].copy()
trips = trips[(trips["fare_amount"].isna()) | (trips["fare_amount"] >= 0)].copy()
trips = trips[~((trips["distance_km"] == 0) & (trips["fare_amount"] > 0))].copy()

## 6. Mean imputation for rider numeric columns

In [ ]:
mean_cols = ["age", "avg_rating"]
mean_imp = SimpleImputer(strategy="mean")
riders[mean_cols] = mean_imp.fit_transform(riders[mean_cols])

## 7. Most-frequent imputation for categorical values

In [ ]:
cat_imp = SimpleImputer(strategy="most_frequent")
for frame, cols in [(riders,["gender","city"]),(trips,["payment_mode","zone"]),(zones,["zone_type"])]:
    frame[cols] = cat_imp.fit_transform(frame[cols])

## 8. KNN imputation: distance, duration, fare

In [ ]:
knn_cols = ["distance_km", "duration_min", "fare_amount"]
knn = KNNImputer(n_neighbors=5)
trips[knn_cols] = knn.fit_transform(trips[knn_cols])

## 9. Convert dates

In [ ]:
riders["signup_date"] = pd.to_datetime(riders["signup_date"], errors="coerce", format="mixed")
trips["ride_date"] = pd.to_datetime(trips["ride_date"], errors="coerce", format="mixed")
print(trips["ride_date"].head())

## 10. Z-score outlier detection (fare, distance)

In [ ]:
fare_z = np.abs(zscore(trips["fare_amount"], nan_policy="omit"))
distance_z = np.abs(zscore(trips["distance_km"], nan_policy="omit"))
print("Fare outliers before:", (fare_z > 3).sum())
print("Distance outliers before:", (distance_z > 3).sum())

## 11. IQR outlier detection for duration

In [ ]:
q1 = trips["duration_min"].quantile(.25)
q3 = trips["duration_min"].quantile(.75)
iqr = q3 - q1
duration_before = ((trips["duration_min"] < q1-1.5*iqr) | (trips["duration_min"] > q3+1.5*iqr)).sum()
print("Duration outliers before:", duration_before)

## 12. Winsorization: cap extreme fare and distance at 1st/99th percentiles

In [ ]:
before_stats = trips[["fare_amount","distance_km","duration_min"]].describe().loc[["mean","std","min","max"]]
for col in ["fare_amount","distance_km"]:
    trips[col] = trips[col].clip(trips[col].quantile(.01), trips[col].quantile(.99))
print("Before:")
display(before_stats)
print("After:")
display(trips[["fare_amount","distance_km","duration_min"]].describe().loc[["mean","std","min","max"]])

## 13. Cap duration using IQR bounds

In [ ]:
trips["duration_min"] = trips["duration_min"].clip(lower=max(0,q1-1.5*iqr), upper=q3+1.5*iqr)

## 14. Outlier comparison

In [ ]:
fare_after = (np.abs(zscore(trips["fare_amount"])) > 3).sum()
distance_after = (np.abs(zscore(trips["distance_km"])) > 3).sum()
q1a,q3a = trips["duration_min"].quantile([.25,.75])
duration_after = ((trips["duration_min"] < q1a-1.5*(q3a-q1a)) | (trips["duration_min"] > q3a+1.5*(q3a-q1a))).sum()
outlier_summary = pd.DataFrame({"Before": [(fare_z>3).sum(),(distance_z>3).sum(),duration_before], "After":[fare_after,distance_after,duration_after]},index=["Fare Z-score","Distance Z-score","Duration IQR"])
display(outlier_summary)

## 15. Datetime features (hour unavailable in source)

In [ ]:
trips["day_of_week"] = trips["ride_date"].dt.dayofweek
trips["month"] = trips["ride_date"].dt.month
trips["hour"] = np.nan  # Only date is recorded; cannot infer hour
print(trips[["ride_date","hour","day_of_week","month"]].head())

## 16. Label encode gender

In [ ]:
le = LabelEncoder()
riders["gender_encoded"] = le.fit_transform(riders["gender"])
display(riders[["gender","gender_encoded"]].head())

## 17. Traffic levels (illustrative thresholds from numeric traffic_index)

In [ ]:
zones["traffic_level"] = pd.cut(zones["traffic_index"],bins=[-np.inf,1,2,np.inf],labels=["Low","Medium","High"])
ordinal = OrdinalEncoder(categories=[["Low","Medium","High"]])
zones["traffic_encoded"] = ordinal.fit_transform(zones[["traffic_level"]]).astype(int)
display(zones[["traffic_index","traffic_level","traffic_encoded"]])

## 18. Merge all three datasets

In [ ]:
df = trips.merge(riders,on="rider_id",how="left",validate="many_to_one")
df = df.merge(zones,left_on="zone",right_on="zone_name",how="left",validate="many_to_one")
display(df.head())

## 19. One-hot encode payment mode and zone name

In [ ]:
df = pd.get_dummies(df,columns=["payment_mode","zone_name"],dtype=int)

## 20. Ride frequency binning (based on observed trip logs)

In [ ]:
df["ride_count"] = df.groupby("rider_id")["trip_id"].transform("count")
df["ride_frequency"] = pd.cut(df["ride_count"],bins=[0,5,15,np.inf],labels=["Low","Medium","High"])
display(df[["rider_id","ride_count","ride_frequency"]].head())

## 21. Log and square-root transformations

In [ ]:
df["fare_log"] = np.log1p(df["fare_amount"])
df["distance_log"] = np.log1p(df["distance_km"])
df["duration_sqrt"] = np.sqrt(df["duration_min"].clip(lower=0))

## 22. Scaling: before statistics

In [ ]:
scale_cols = ["fare_amount","distance_km","duration_min"]
display(df[scale_cols].agg(["mean","std","min","max"]))

## 23. StandardScaler

In [ ]:
standard = StandardScaler()
standard_values = standard.fit_transform(df[scale_cols])
for i,col in enumerate(scale_cols):
    df[col+"_standard"] = standard_values[:,i]
display(df[[c+"_standard" for c in scale_cols]].agg(["mean","std","min","max"]))

## 24. MinMaxScaler

In [ ]:
minmax = MinMaxScaler()
minmax_values = minmax.fit_transform(df[scale_cols])
for i,col in enumerate(scale_cols):
    df[col+"_minmax"] = minmax_values[:,i]
display(df[[c+"_minmax" for c in scale_cols]].agg(["mean","std","min","max"]))

## 25. Feature: average ride distance per rider

In [ ]:
df["avg_ride_distance"] = df.groupby("rider_id")["distance_km"].transform("mean")

## 26. Feature: average ride fare per rider

In [ ]:
df["avg_ride_fare"] = df.groupby("rider_id")["fare_amount"].transform("mean")

## 27. Feature: peak hour (unknown, do not invent time)

In [ ]:
df["is_peak_hour"] = pd.Series(pd.NA,index=df.index,dtype="Int64")
print("Peak-hour data unavailable: ride_date has no time component")

## 28. Feature: days since signup

In [ ]:
today = pd.Timestamp.today().normalize()
df["days_since_signup"] = (today - df["signup_date"]).dt.days

## 29. Feature: cancellation rate from rider account totals

In [ ]:
df["ride_cancellation_rate"] = df["cancelled_rides"] / df["total_rides"].replace(0,np.nan)
df["ride_cancellation_rate"] = df["ride_cancellation_rate"].fillna(0)

## 30. Feature: fare-per-km threshold and existing surge flag

In [ ]:
df["fare_per_km"] = df["fare_amount"] / df["distance_km"].replace(0,np.nan)
threshold = df["fare_per_km"].median() * 1.5
df["surge_by_threshold"] = (df["fare_per_km"] > threshold).astype(int)
# Existing surge_flag is retained as the recorded flag; derived rule is a separate column.
display(df[["surge_flag","surge_by_threshold","fare_per_km"]].head())

## 31. Final quality checks and summary

In [ ]:
rows_after = len(df)
missing_after = df.drop(columns=["hour","is_peak_hour"]).isna().sum().sum()
summary = pd.DataFrame({"Metric":["Trip rows","Missing values (excluding unavailable hour fields)","Fare outliers (Z)","Distance outliers (Z)","Duration outliers (IQR)","New engineered features"],"Before":[rows_before,missing_before,(fare_z>3).sum(),(distance_z>3).sum(),duration_before,0],"After":[rows_after,missing_after,fare_after,distance_after,duration_after,7]})
display(summary)
print("Missing by column:")
print(df.isna().sum()[df.isna().sum()>0])

## 32. Export final CSV

In [ ]:
df.to_csv("final_prepared_rides_dataset.csv",index=False)
print("Saved final_prepared_rides_dataset.csv")

## 33. Bonus: ride demand by day of week (hourly data not available)

In [ ]:
df["day_of_week"].value_counts().sort_index().plot(kind="bar",title="Ride Demand by Day of Week")
plt.xlabel("Day (Monday=0)")
plt.ylabel("Trips")
plt.tight_layout()
plt.savefig("ride_demand_by_weekday.png")
plt.show()

## 34. Bonus: surge vs no surge

In [ ]:
df["surge_flag"].value_counts().sort_index().plot(kind="bar")
plt.title("Recorded Surge vs No-Surge Trips")
plt.tight_layout()
plt.savefig("surge_patterns.png")
plt.show()